# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we explore the record sets, their `@id`s, and associated field `@id`s as described in the Croissant schema.

In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.record_sets())
print(f"Record sets available in this dataset (by @id):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (by @id):")
        for field in fields:
            # Each field is a dict with '@id' and maybe 'name'. Print both if present
            if isinstance(field, dict):
                name = field.get('name', None)
                print(f"    - {field['@id']}" + (f" (name: {name})" if name else ""))
            else:
                print(f"    - {field}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, all record sets are loaded into pandas DataFrames using their `@id` variables for future manipulation.

In [ ]:
# If there are no record sets, dataset.records() returns everything.
# But most structured scientific datasets will have one main RecordSet.
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns for record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We demonstrate these actions below on one numeric field from the first record set. Update the field `@id` as needed.

In [ ]:
# Choose a record set and numeric field for processing (replace @ids as needed)

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to guess a numeric field by looking for float/int columns
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if not numeric_cols:
        # Try to convert possible columns that should be numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}\n")

        threshold = df[numeric_field].mean() if df[numeric_field].count() > 0 else 0
        if threshold == 0:
            threshold = 1
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a groupable field (categorical)
        cat_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        # Prefer something not the index
        for f in cat_fields:
            if f != numeric_field and df[f].nunique() > 1 and df[f].nunique() < 25:
                group_field = f
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No record sets for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we display the distribution of the selected numeric field and (if possible) its values grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        if group_field in df.columns:
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()
else:
    print("No numeric fields or record sets for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and examined the structure via its Croissant schema.
- Extracted available record sets and fields by their `@id` for transparent referencing and reproducibility.
- Demonstrated filtering, normalization, and grouping operations on one numeric field, and visualized data distributions.
- This notebook provides a template for detailed, reproducible analysis of complex FAIR-structured datasets using `mlcroissant`. Use field and record set `@id`s for further, domain-specific statistical or machine learning analysis.